In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import math
import os
import warnings
import requests
from bs4 import BeautifulSoup
import statsmodels
from statsmodels.stats import proportion
from statsmodels.stats import diagnostic
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import ttest_ind
from io import StringIO

In [33]:
#data from json file

data_folder = './data/newyorker_caption_contest/'
df = pd.read_json(data_folder + 'contests.json')
metadata = df['metadata'].apply(pd.Series)
df = pd.concat([df.drop(columns=['metadata']), metadata], axis=1)

df.head(10)

,contest_id,image,data,num_captions,num_votes,image_locations,image_descriptions,image_uncanny_descriptions,entities,questions
0,510,images/510.jpg,data/510.csv,3905.0,41185.0,[the street],[A man is relaxing on a city street. Others ar...,[A man is just laying in the middle of the sid...,[https://en.wikipedia.org/wiki/Bystander_effec...,[Why is he laying there?]
1,511,images/511.jpg,data/511.csv,3325.0,28205.0,"[the front hard, a residential walkway]",[A man in a winter coat and cap is looking at ...,[It's unusual to see someone holding a snow sh...,"[https://en.wikipedia.org/wiki/Snowball_fight,...",[Is the man overly small or the shovel overly ...
2,512,images/512.jpg,data/512.csv,4399.0,21574.0,"[yoga place, a yoga studio]",[A man and woman are standing facing one anoth...,[Nothing is really out of place in this image....,"[https://en.wikipedia.org/wiki/Rug, https://en...","[Why is the man carrying a huge rug?, Why is t..."
3,513,images/513.jpg,data/513.csv,4141.0,16894.0,"[a workplace, an elevator]",[Three business men are walking down a hall. T...,[A suit case is usually carried by one person ...,[https://en.wikipedia.org/wiki/Worker_cooperat...,[Why is the briefcase big enough for three peo...
4,514,images/514.jpg,data/514.csv,3951.0,95790.0,[plains],[Some cowboys are riding through the desert. T...,[There are rocking horses in place of real hor...,"[https://en.wikipedia.org/wiki/Rocking_horse, ...",[Why is this chase taking place?]
5,515,images/515.jpg,data/515.csv,4270.0,23543.0,"[a railroad, a subway]",[People are laying on the subway tracks as an ...,[They are putting themselves in danger by bein...,"[https://en.wikipedia.org/wiki/Train, https://...","[Why are they bandaged up?, What caused their ..."
6,516,images/516.jpg,data/516.csv,5918.0,24660.0,"[restaurant, a restaurant]",[A couple is having drinks at a table when a c...,[The cinderblock is coming out of nowhere and ...,"[https://en.wikipedia.org/wiki/Safety, https:/...",[Why is a cement block falling from the ceilin...
7,517,images/517.jpg,data/517.csv,7782.0,29913.0,[a courtroom],[A lawyer is with his client in a courtroom. T...,[The client is Death.],"[https://en.wikipedia.org/wiki/Lawyer, https:/...",[How do you arrest Death?]
8,518,images/518.jpg,data/518.csv,4509.0,123937.0,"[top of building, a rooftop]",[A man is crawling across a sandbox. Two kids ...,[There is an adult man in a sandbox for kids. ...,"[https://en.wikipedia.org/wiki/Sandpit, https:...","[Why is the man crawling around the sandbox?, ..."
9,519,images/519.jpg,data/519.csv,4614.0,96184.0,"[woodlands, forest]",[Two pheasants are together watching a hunter....,[Birds would be too scared to stay on the gun....,"[https://en.wikipedia.org/wiki/Bird, https://e...","[Why is the bird on the gun?, Why is the bird ..."


In [66]:
#scrape data from caption contest website

url = "https://nextml.github.io/caption-contest-data/"
soup = BeautifulSoup(requests.get(url).content, 'html.parser')

df_scrape = pd.read_html(StringIO(str(soup.find('table'))))[0]
df_scrape

,Contest Dashboard,Cartoon,Top Rated caption,The New Yorker's winner,Finalists Announced (date of issue),Number of votes
0,895 Dashboard,NaN,“Do you think death rays would be considered e...,NaN,NaN,777820
1,894 Dashboard,NaN,“See how the nose seems to follow you?”,NaN,NaN,878676
2,893 Dashboard,NaN,“The seller isn't willing to come down.”,NaN,NaN,1085745
3,892 Dashboard,NaN,"“Yes, it's called Thesaurus, but you're not re...",NaN,NaN,788251
4,891 Dashboard,NaN,"“Sorry, only the catcher works from home.”",NaN,NaN,928856
...,...,...,...,...,...,...
380,514 Dashboard,NaN,"“No, you grow up”",NaN,2016-04-03 (estimated),192198
381,513 Dashboard,NaN,"“I'm just saying, I can see why the 'brief'cas...",NaN,2016-03-27 (estimated),34013
382,512 Dashboard,NaN,"“We're pretentious, not ostentatious.”",NaN,2016-03-21 (estimated),43120
383,511 Dashboard,NaN,“I'm hourly.”,NaN,NaN,56660


In [67]:
#clean df_scrape columns

df_scrape['Contest Dashboard'] = df_scrape['Contest Dashboard'].str.replace(' Dashboard', '').str.strip()
df_scrape.rename(columns={'Finalists Announced (date of issue)': 'Date'}, inplace=True)
df_scrape['Date'] = df_scrape['Date'].str.replace(' (estimated)', '').str.strip()
df_scrape.drop(columns=['Cartoon'], inplace=True)
df_scrape = df_scrape.iloc[::-1].reset_index(drop=True)
df_scrape.head(100)

,Contest Dashboard,Top Rated caption,The New Yorker's winner,Date,Number of votes
0,510,“I'm a congressman--obstruction is my job.”,NaN,NaN,82627
1,511,“I'm hourly.”,NaN,NaN,56660
2,512,"“We're pretentious, not ostentatious.”",NaN,2016-03-21,43120
3,513,"“I'm just saying, I can see why the 'brief'cas...",NaN,2016-03-27,34013
4,514,"“No, you grow up”",NaN,2016-04-03,192198
...,...,...,...,...,...
95,606,“As senior squeegee-man during flight simulati...,NaN,NaN,46951
96,607,“My only regret is that he never got around to...,NaN,NaN,1026981
97,608,“The hard part was transposing from C major to...,NaN,NaN,957697
98,609,“He really did appoint Chris Christie as Secre...,NaN,NaN,1046199


In [68]:
#check if all expected contest IDs are present
expected_ids = list(range(510, 896))
missing_ids_df = pd.Series(expected_ids)[~pd.Series(expected_ids).isin(df['contest_id'])]
missing_ids_scrape = pd.Series(expected_ids)[~pd.Series(expected_ids).isin(df_scrape['Contest Dashboard'].astype(int))]

print('Missing data from df: ',missing_ids_df.tolist())
print('Missing data from df_scrape: ',missing_ids_scrape.tolist())

Missing data from df:  [525, 540]
Missing data from df_scrape:  [525]


what to do now ? shift all ids or not ?

In [69]:
from datetime import timedelta

df_scrape['Date'] = pd.to_datetime(df_scrape['Date'], errors='coerce')

# Assume contest happens each week and fill missing dates
for i in range(1, len(df_scrape)):
    if pd.isna(df_scrape.loc[i, 'Date']):
        df_scrape.loc[i, 'Date'] = df_scrape.loc[i - 1, 'Date'] + timedelta(days=7)

for i in range(len(df_scrape)-2, -1, -1):
    if pd.isna(df_scrape.loc[i, 'Date']):
        df_scrape.loc[i, 'Date'] = df_scrape.loc[i+1, 'Date'] - timedelta(days=7)
    
df_scrape

,Contest Dashboard,Top Rated caption,The New Yorker's winner,Date,Number of votes
0,510,“I'm a congressman--obstruction is my job.”,NaN,2016-03-07,82627
1,511,“I'm hourly.”,NaN,2016-03-14,56660
2,512,"“We're pretentious, not ostentatious.”",NaN,2016-03-21,43120
3,513,"“I'm just saying, I can see why the 'brief'cas...",NaN,2016-03-27,34013
4,514,"“No, you grow up”",NaN,2016-04-03,192198
...,...,...,...,...,...
380,891,"“Sorry, only the catcher works from home.”",NaN,2023-08-21,928856
381,892,"“Yes, it's called Thesaurus, but you're not re...",NaN,2023-08-28,788251
382,893,“The seller isn't willing to come down.”,NaN,2023-09-04,1085745
383,894,“See how the nose seems to follow you?”,NaN,2023-09-11,878676


In [91]:
df_scrape['Contest Dashboard'] = df_scrape['Contest Dashboard'].astype(int)
df_complete = pd.merge(df, df_scrape, left_on='contest_id', right_on='Contest Dashboard')

df_complete.drop(columns=['image'], inplace=True)
df_complete.drop(columns=['data'], inplace=True)
df_complete.drop(columns=['Contest Dashboard'], inplace=True)
df_complete.drop(columns=['num_votes'], inplace=True) #better to keep Number of Votes from the official website
df_complete.head(10)

,contest_id,num_captions,image_locations,image_descriptions,image_uncanny_descriptions,entities,questions,Top Rated caption,The New Yorker's winner,Date,Number of votes
0,510,3905.0,[the street],[A man is relaxing on a city street. Others ar...,[A man is just laying in the middle of the sid...,[https://en.wikipedia.org/wiki/Bystander_effec...,[Why is he laying there?],“I'm a congressman--obstruction is my job.”,NaN,2016-03-07,82627
1,511,3325.0,"[the front hard, a residential walkway]",[A man in a winter coat and cap is looking at ...,[It's unusual to see someone holding a snow sh...,"[https://en.wikipedia.org/wiki/Snowball_fight,...",[Is the man overly small or the shovel overly ...,“I'm hourly.”,NaN,2016-03-14,56660
2,512,4399.0,"[yoga place, a yoga studio]",[A man and woman are standing facing one anoth...,[Nothing is really out of place in this image....,"[https://en.wikipedia.org/wiki/Rug, https://en...","[Why is the man carrying a huge rug?, Why is t...","“We're pretentious, not ostentatious.”",NaN,2016-03-21,43120
3,513,4141.0,"[a workplace, an elevator]",[Three business men are walking down a hall. T...,[A suit case is usually carried by one person ...,[https://en.wikipedia.org/wiki/Worker_cooperat...,[Why is the briefcase big enough for three peo...,"“I'm just saying, I can see why the 'brief'cas...",NaN,2016-03-27,34013
4,514,3951.0,[plains],[Some cowboys are riding through the desert. T...,[There are rocking horses in place of real hor...,"[https://en.wikipedia.org/wiki/Rocking_horse, ...",[Why is this chase taking place?],"“No, you grow up”",NaN,2016-04-03,192198
5,515,4270.0,"[a railroad, a subway]",[People are laying on the subway tracks as an ...,[They are putting themselves in danger by bein...,"[https://en.wikipedia.org/wiki/Train, https://...","[Why are they bandaged up?, What caused their ...",“Tell me about your childhood very quickly.”,NaN,2016-04-10,47448
6,516,5918.0,"[restaurant, a restaurant]",[A couple is having drinks at a table when a c...,[The cinderblock is coming out of nowhere and ...,"[https://en.wikipedia.org/wiki/Safety, https:/...",[Why is a cement block falling from the ceilin...,“I have no concrete plans for the rest of the ...,NaN,2016-04-17,49365
7,517,7782.0,[a courtroom],[A lawyer is with his client in a courtroom. T...,[The client is Death.],"[https://en.wikipedia.org/wiki/Lawyer, https:/...",[How do you arrest Death?],“Who'd have thought they'd get you for tax eva...,NaN,2016-04-24,59717
8,518,4509.0,"[top of building, a rooftop]",[A man is crawling across a sandbox. Two kids ...,[There is an adult man in a sandbox for kids. ...,"[https://en.wikipedia.org/wiki/Sandpit, https:...","[Why is the man crawling around the sandbox?, ...",“This saddest part is he's going in the wrong ...,NaN,2016-05-01,248180
9,519,4614.0,"[woodlands, forest]",[Two pheasants are together watching a hunter....,[Birds would be too scared to stay on the gun....,"[https://en.wikipedia.org/wiki/Bird, https://e...","[Why is the bird on the gun?, Why is the bird ...","“He's pro-gun, but I like his stance on migrat...",NaN,2016-05-08,191935


In [92]:
def separate(df, col):
    '''
    Create a new dataframe by exploding and isolating a specific column.
    
    Parameters:
    df (pd.DataFrame): The original dataframe.
    col (str): The column to explode and isolate.
    
    Return:
    pd.DataFrame: A new dataframe with 'contest_id' matching the specified column.
    '''
    
    df_new = df.explode(col)
    return df_new[['contest_id', col]]

In [93]:
df_complete.columns

Index(['contest_id', 'num_captions', 'image_locations', 'image_descriptions',
       'image_uncanny_descriptions', 'entities', 'questions',
       'Top Rated caption', 'The New Yorker's winner', 'Date',
       'Number of votes'],
      dtype='object')

In [96]:
#separate dataframes
df_base = df_complete[['contest_id', 'num_captions', 'Number of votes', 'Date', 'Top Rated caption', "The New Yorker's winner"]]
df_locations = separate(df_complete, 'image_locations')
df_descriptions = separate(df_complete, 'image_descriptions')
df_uncanny = separate(df_complete, 'image_uncanny_descriptions')
df_entities = separate(df_complete, 'entities')
df_questions = separate(df_complete, 'questions')

df_base.head(20)

,contest_id,num_captions,Number of votes,Date,Top Rated caption,The New Yorker's winner
0,510,3905.0,82627,2016-03-07,“I'm a congressman--obstruction is my job.”,NaN
1,511,3325.0,56660,2016-03-14,“I'm hourly.”,NaN
2,512,4399.0,43120,2016-03-21,"“We're pretentious, not ostentatious.”",NaN
3,513,4141.0,34013,2016-03-27,"“I'm just saying, I can see why the 'brief'cas...",NaN
4,514,3951.0,192198,2016-04-03,"“No, you grow up”",NaN
5,515,4270.0,47448,2016-04-10,“Tell me about your childhood very quickly.”,NaN
6,516,5918.0,49365,2016-04-17,“I have no concrete plans for the rest of the ...,NaN
7,517,7782.0,59717,2016-04-24,“Who'd have thought they'd get you for tax eva...,NaN
8,518,4509.0,248180,2016-05-01,“This saddest part is he's going in the wrong ...,NaN
9,519,4614.0,191935,2016-05-08,"“He's pro-gun, but I like his stance on migrat...",NaN


In [84]:
#df_entities : clean column

df_entities['entities'] = df_entities['entities'].str.replace('https://en.wikipedia.org/wiki/', '').str.strip()
df_entities['entities'] = df_entities['entities'].str.replace('_', ' ').str.strip()
df_entities

,contest_id,entities
0,510,Bystander effect
0,510,Crowd
1,511,Snowball fight
1,511,Winter
1,511,Snow removal
...,...,...
379,891,NaN
380,892,NaN
381,893,NaN
382,894,NaN
